# **Machine Learning**

### **Import**

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

load_dotenv()
db = MongoClient(os.getenv("MONGO_URI"))["cinema"]

In [ ]:
films = pd.DataFrame(db["films"].find(
    {"budget": {"$gte": 10_000}, "recettes": {"$gte": 10_000}},
    {"titre": 1, "budget": 1, "recettes": 1, "duree_min": 1, "genres": 1,
     "date_sortie": 1, "annee": 1, "langue_originale": 1, "saga": 1}
))
films = films.dropna(subset=["duree_min", "date_sortie"])
print(len(films), "films")
print("Colonnes :", list(films.columns))

In [ ]:
films["log_recettes"] = np.log10(films["recettes"])
films["log_budget"] = np.log10(films["budget"])
films["mois"] = pd.to_datetime(films["date_sortie"]).dt.month
films["saga"] = films["saga"].notna().astype(int)
films["anglais"] = (films["langue_originale"] == "en").astype(int)

genres = films["genres"].str.join("|").str.get_dummies()
genres.columns = ["genre_" + g for g in genres.columns]
films = pd.concat([films, genres], axis=1)

variables = ["log_budget", "duree_min", "mois", "annee", "saga", "anglais"] + list(genres.columns)
print(len(variables), "variables explicatives")

In [ ]:
X = films[variables]
y = films["log_recettes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Entraînement :", len(X_train), "films | Test :", len(X_test), "films")

In [ ]:
modeles = {
    "Régression linéaire": LinearRegression(),
    "Forêt aléatoire": RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                             random_state=42, n_jobs=-1),
}
scores = []
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    scores.append({"modèle": nom, "R²": round(r2_score(y_test, pred), 3),
                   "erreur typique (×)": round(10 ** mae, 2)})
pd.DataFrame(scores)

In [ ]:
print("Variables suspectes :", [v for v in variables if "recette" in v or "vote" in v or "note" in v])
print(X.corrwith(y).abs().sort_values(ascending=False).head(5))